# 01 - Exploratory Data Analysis (EDA)
===

Maritime Conflict Intelligence System (MCIS) ようこそ!

This notebook performs initial data exploration and visualization.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import warnings

warnings.filterwarnings('ignore')

plt.rcParams.update({
    'figure.dpi': 150,
    'figure.figsize': (10, 6),
    'font.size': 11,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'axes.grid': True,
    'grid.alpha': 0.3,
})
sns.set_style('whitegrid')

DATA_DIR = Path('./data/processed')
OUTPUT_DIR = Path('./outputs/figures/eda')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Vessel type mapping
VESSEL_TYPE_MAP = {
    0: "Not Available", 30: "Fishing", 31: "Towing",
    32: "Towing Large", 33: "Dredging", 34: "Diving",
    35: "Military", 36: "Sailing", 37: "Pleasure",
    50: "Pilot", 51: "SAR", 52: "Tug",
    55: "Law Enforcement", 60: "Passenger",
    70: "Cargo", 80: "Tanker", 90: "Other"
}

## Load Data

In [ ]:
df = pd.read_parquet(DATA_DIR / 'ais_features.parquet')
print(f'Records: {len(df):,}')
print(f'Columns: {len(df.columns)}')
print(f'Memory: {df.memory_usage(deep=True).sum() / 1e6:.1f} MB')
df.head(3)

## Data Overview

In [ ]:
print("\n=== Column Types ===")
print(df.dtypes.value_counts())

print("\n=== Numeric Summary ===")
df.describe()

## Missing Values Analysis

In [ ]:
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)
missing_df = pd.DataFrame({'missing': missing, 'pct': missing_pct})
missing_df = missing_df[missing_df['missing'] > 0].sort_values('pct', ascending=False)
print(f"Columns with missing: {len(missing_df)}")
missing_df.head(10)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
sns.barplot(data=missing_df, y=missing_df.index, x='pct', ax=ax, palette='viridis')
ax.set_xlabel('Missing %')
ax.set_title('Missing Values by Column')
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'missing_values.png', dpi=300, bbox_inches='tight')
plt.show()

## Vessel Types Distribution

In [ ]:
# Map vessel types
if 'VesselType' in df.columns:
    df['VesselTypeName'] = df['VesselType'].map(VESSEL_TYPE_MAP).fillna('Unknown')

vessel_counts = df['VesselType'].value_counts().head(12)
fig, ax = plt.subplots(figsize=(10, 5))
sns.barplot(x=vessel_counts.values, y=vessel_counts.index, ax=ax, palette='Blues_d')
ax.set_xlabel('Count')
ax.set_ylabel('Vessel Type Code')
ax.set_title('Top 12 Vessel Types')
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'vessel_types.png', dpi=300, bbox_inches='tight')
plt.show()

## Speed Distribution Analysis

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

sog = df['SOG'].dropna()
sns.histplot(x=sog, bins=50, ax=axes[0], color='teal', kde=True)
axes[0].set_xlabel('Speed Over Ground (knots)')
axes[0].set_title(f'Speed Distribution (n={len(sog):,})')
axes[0].axvline(sog.mean(), color='red', linestyle='--', label=f'Mean: {sog.mean():.1f}')
axes[0].legend()

sns.boxplot(data=df, y='SOG', ax=axes[1], palette='viridis')
axes[1].set_ylabel('Speed (knots)')
axes[1].set_title('Speed Boxplot')

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'speed_distribution.png', dpi=300, bbox_inches='tight')
plt.show()

print(f"Speed Stats: mean={sog.mean():.2f}, std={sog.std():.2f}, min={sog.min():.2f}, max={sog.max():.2f}")

## Geographic Distribution

In [ ]:
# Sample for visualization if too large
plot_df = df.sample(n=min(50000, len(df)), random_state=42) if len(df) > 50000 else df

fig, ax = plt.subplots(figsize=(12, 7))
scatter = ax.scatter(
    plot_df['LON'], plot_df['LAT'],
    c=plot_df['SOG'], cmap='viridis',
    alpha=0.5, s=15, edgecolor='none'
)
cbar = plt.colorbar(scatter)
cbar.set_label('Speed (knots)')
ax.set_xlabel('Longitude')
ax.set_ylabel('Latitude')
ax.set_title('Vessel Traffic Geographic Distribution')
ax.set_xlim(-180, 180)
ax.set_ylim(-90, 90)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'geographic_distribution.png', dpi=300, bbox_inches='tight')
plt.show()

## Temporal Analysis

In [ ]:
if 'BaseDateTime' in df.columns:
    df['hour'] = df['BaseDateTime'].dt.hour
    df['date'] = df['BaseDateTime'].dt.date
    df['dayofweek'] = df['BaseDateTime'].dt.day_name()

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

hourly = df.groupby('hour')['MMSI'].nunique()
sns.barplot(x=hourly.index, y=hourly.values, ax=axes[0], palette='coral')
axes[0].set_xlabel('Hour of Day (UTC)')
axes[0].set_ylabel('Unique Vessels')
axes[0].set_title('Hourly Vessel Activity')
axes[0].set_xticks(range(0, 24, 2))

daily = df.groupby('date')['MMSI'].nunique()
sns.lineplot(x=daily.index, y=daily.values, ax=axes[1], marker='o', color='teal', alpha=0.7)
axes[1].set_xlabel('Date')
axes[1].set_ylabel('Unique Vessels')
axes[1].set_title('Daily Vessel Activity')
axes[1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'temporal_distribution.png', dpi=300, bbox_inches='tight')
plt.show()

## Conflict Zone Distribution

In [ ]:
zone_counts = df['conflict_zone_name'].value_counts()
zone_counts = zone_counts[zone_counts.index != 'none']

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

colors = plt.cm.Spectral(np.linspace(0, 1, len(zone_counts)))
axes[0].pie(zone_counts.values, labels=zone_counts.index, autopct='%1.1f%%', 
           colors=colors, startangle=90)
axes[0].set_title('Conflict Zone Proportions')

sns.barplot(x=zone_counts.values, y=zone_counts.index, ax=axes[1], palette='steelblue')
axes[1].set_xlabel('Records')
axes[1].set_ylabel('Conflict Zone')
axes[1].set_title('Records by Conflict Zone')

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'conflict_zones.png', dpi=300, bbox_inches='tight')
plt.show()

## Correlation Analysis

In [ ]:
numeric_cols = ['SOG', 'COG', 'Heading', 'VesselType', 'Length', 'Width', 'Draft']
numeric_cols = [c for c in numeric_cols if c in df.columns]
corr = df[numeric_cols].corr()

fig, ax = plt.subplots(figsize=(10, 8))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', center=0, ax=ax, 
            square=True, linewidths=0.5, annot_kws={'size': 9})
ax.set_title('Feature Correlation Matrix')
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'correlation_matrix.png', dpi=300, bbox_inches='tight')
plt.show()

## Summary Statistics

In [ ]:
print("="*50)
print("EDA SUMMARY")
print("="*50)
print(f"Total Records: {len(df):,}")
print(f"Unique Vessels: {df['MMSI'].nunique():,}")
print(f"Date Range: {df['BaseDateTime'].min()} to {df['BaseDateTime'].max()}")
print(f"Geographic Bounds: LON [{df['LON'].min():.1f}, {df['LON'].max():.1f}], LAT [{df['LAT'].min():.1f}, {df['LAT'].max():.1f}]")
print(f"Mean Speed: {df['SOG'].mean():.2f} knots")
print(f"Conflict Zones: {len(zone_counts)}")
print("="*50)
print(f"Output saved to: {OUTPUT_DIR}")